In [37]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Spark Job Progress Monitor already enabled


In [49]:
#Step2
Epilepsy_Combined = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-smallset-finalF1_Numbered_NoNullStr.parquet")

In [50]:
#Step3
balanced_df = Epilepsy_Combined.drop("personid")

In [51]:
import numpy as np
# setting random seed for notebook reproducability
rnd_seed=23
np.random.seed=rnd_seed
np.random.set_state=rnd_seed

In [52]:
train_data, valid_data, test_data = balanced_df.randomSplit([.7,.2,.1], seed=rnd_seed)

In [53]:
train_data = train_data.withColumn('label',train_data.label.cast('double'))

In [54]:
#Step5
columns = train_data.columns
feature_cols = columns
feature_cols.remove('label')
target_col = ['label']

In [55]:
#Step6
###########################Included Standard Scalar #############################################################
from pyspark.sql import DataFrame
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, VectorSizeHint, StandardScaler

# Step 1: Define vector columns and non-vector columns based on schema
vector_cols = [col_name for col_name, dtype in train_data.dtypes if 'vector' in dtype]
non_vector_cols = [col_name for col_name in feature_cols if col_name not in vector_cols]

# Step 2: Function to apply VectorSizeHint and return size hint stages for the pipeline
def get_vector_size_hint_stage(data, col_name):
    # Sample a small portion of the data to determine vector size
    sample_fraction = 0.0001  # Using 0.01% of the data for sampling
    sampled_df = data.select(col_name).sample(False, sample_fraction).limit(1)
    sample_row = sampled_df.take(1)

    if sample_row:
        vector_size = len(sample_row[0][col_name])
        print(f"Column '{col_name}' vector size: {vector_size}")
        # Return a VectorSizeHint stage for the pipeline if vector size is valid
        if vector_size > 0:
            return VectorSizeHint(inputCol=col_name, size=vector_size)
    else:
        print(f"No sample found for column '{col_name}'. Skipping VectorSizeHint.")
    
    return None

# Step 3: Create a list of VectorSizeHint stages for each vector column
vector_size_hint_stages = []
for col_name in vector_cols:
    print(f"Getting VectorSizeHint for vector column: '{col_name}'")
    size_hint_stage = get_vector_size_hint_stage(train_data, col_name)
    if size_hint_stage:
        vector_size_hint_stages.append(size_hint_stage)

# Step 4: Combine vector and non-vector columns into a single list for VectorAssembler
final_input_cols = vector_cols + non_vector_cols
print("Final input columns for feature assembly:", final_input_cols)

# Step 5: Assemble final features column using VectorAssembler
final_assembler = VectorAssembler(inputCols=final_input_cols, outputCol="features")

# Step 6: Initialize StandardScaler
standardScaler = StandardScaler(inputCol="features", outputCol="features_scaled")

# Step 7: Create a pipeline with VectorSizeHint stages, VectorAssembler, and StandardScaler
pipeline_stages = vector_size_hint_stages + [final_assembler, standardScaler]
pipeline_final = Pipeline(stages=pipeline_stages)

# Step 8: Fit the pipeline on train_data
model_final = pipeline_final.fit(train_data)
print("Pipeline fitting done.")

# Step 9: Transform train, valid, and test datasets using the fitted pipeline
train_data = model_final.transform(train_data)
valid_data = model_final.transform(valid_data)
test_data = model_final.transform(test_data)
print("Pipeline transformation done.")

Getting VectorSizeHint for vector column: 'gender_onehot'
Column 'gender_onehot' vector size: 4
Getting VectorSizeHint for vector column: 'race_onehot'
Column 'race_onehot' vector size: 7
Final input columns for feature assembly: ['gender_onehot', 'race_onehot', 'age_of_TBI_diagnosis', 'MedicalHistory', 'Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B

Pipeline fitting done.
Pipeline transformation done.


In [56]:
#Step7
train_data = train_data.withColumn('label',train_data.label.cast('double'))
valid_data = valid_data.withColumn('label',valid_data.label.cast('double'))
test_data = test_data.withColumn('label',test_data.label.cast('double'))

In [57]:
from pyspark.sql.functions import col

# Count the number of 0's and 1's in the label column
count_zeros = train_data.filter(col('label') == 0).count()
count_ones = train_data.filter(col('label') == 1).count()

# Separate the majority and minority classes
majority_class_df = train_data.filter(col('label') == 0)
minority_class_df = train_data.filter(col('label') == 1)

# Check if we need to upsample or downsample
if count_zeros > count_ones:
    # Upsample the minority class
    upsample_ratio = count_zeros // count_ones  # Calculate integer part of ratio
    remaining_samples_fraction = (count_zeros % count_ones) / count_ones  # Remaining fraction

    # Duplicate the minority class to match the majority class count
    upsampled_minority_class_df = minority_class_df
    for _ in range(upsample_ratio - 1):  # -1 because we already have one instance
        upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df)

    # Add the remaining samples to reach exact count
    upsampled_minority_class_df = upsampled_minority_class_df.union(
        minority_class_df.sample(withReplacement=True, fraction=remaining_samples_fraction)
    )

    # Combine with majority class
    train_data_balanced = majority_class_df.union(upsampled_minority_class_df)

else:
    # Downsample the majority class if count_ones > count_zeros
    downsample_fraction = count_ones / count_zeros
    downsampled_majority_class_df = majority_class_df.sample(withReplacement=False, fraction=downsample_fraction)

    # Combine with minority class
    train_data_balanced = downsampled_majority_class_df.union(minority_class_df)

# Display the counts after balancing
print("Number of 0's in the balanced DataFrame:", train_data_balanced.filter(col('label') == 0).count())
print("Number of 1's in the balanced DataFrame:", train_data_balanced.filter(col('label') == 1).count())

Number of 0's in the balanced DataFrame: 70414
Number of 1's in the balanced DataFrame: 70268


In [58]:
# Number of partitions to repartition into (adjust based on your data size and cluster configuration)
num_partitions = train_data.rdd.getNumPartitions()

# Repartition the DataFrame before processing
try:
    train_data_balanced = train_data_balanced.repartition(num_partitions)
    print(f"DataFrame repartitioned into {num_partitions} partitions successfully.")
except Exception as e:
    print(f"Error during repartitioning: {e}")
    
# Number of partitions to repartition into (adjust based on your data size and cluster configuration)
num_partitions1 = valid_data.rdd.getNumPartitions()

# Repartition the DataFrame before processing
try:
    valid_data = valid_data.repartition(num_partitions1)
    print(f"DataFrame repartitioned into {num_partitions1} partitions successfully.")
except Exception as e:
    print(f"Error during repartitioning: {e}")
    
# Number of partitions to repartition into (adjust based on your data size and cluster configuration)
num_partitions2 = test_data.rdd.getNumPartitions()

# Repartition the DataFrame before processing
try:
    test_data = test_data.repartition(num_partitions2)
    print(f"DataFrame repartitioned into {num_partitions2} partitions successfully.")
except Exception as e:
    print(f"Error during repartitioning: {e}")

DataFrame repartitioned into 10 partitions successfully.
DataFrame repartitioned into 10 partitions successfully.
DataFrame repartitioned into 10 partitions successfully.


In [ ]:
import os
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col

# Define the GBT model with initial parameters
gbt = GBTClassifier(labelCol="label", featuresCol="features_scaled", maxIter=10, maxDepth=5)

# Define a global binary evaluator for areaUnderROC
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Adaptive Early Stopping Function for GBT
def adaptive_early_stopping(classifier, train_data, valid_data, patience=5, max_iter_increment=10, max_iter_cap=100):
    """
    Train a classifier with adaptive early stopping by incrementally increasing maxIter.
    """
    best_model = None
    best_auc = 0.0
    best_iter = 0
    patience_counter = 0
    current_max_iter = max_iter_increment

    while current_max_iter <= max_iter_cap:
        classifier.setMaxIter(current_max_iter)
        print(f"\nTraining with maxIter={current_max_iter}...")

        model = classifier.fit(train_data)
        valid_predictions = model.transform(valid_data)
        validation_auc = evaluator.evaluate(valid_predictions)

        print(f"Validation AUC with maxIter={current_max_iter}: {validation_auc}")

        if validation_auc > best_auc:
            best_auc = validation_auc
            best_model = model
            best_iter = current_max_iter
            patience_counter = 0
        else:
            patience_counter += 1

        # Early stopping condition
        if patience_counter >= patience:
            print(f"Early stopping triggered. Best maxIter: {best_iter} with Validation AUC: {best_auc}")
            break

        # Increment maxIter for the next iteration
        current_max_iter += max_iter_increment

    return best_model, best_auc

# Define predictions path
predictions_path = "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/checkpoint4"

# Check if the directory exists
if not os.path.exists(predictions_path):
    # Training and Adaptive Early Stopping
    print("\nTraining GBT with Adaptive Early Stopping...")
    best_gbt_model, best_gbt_auc = adaptive_early_stopping(
        gbt, train_data_balanced, valid_data, patience=3, max_iter_increment=10, max_iter_cap=100
    )
    print(f"Best AUC for GBT: {best_gbt_auc}")

    # Evaluate the best model on test data
    test_predictions_gbt = best_gbt_model.transform(test_data)

    # Save predictions to Parquet
    test_predictions_gbt.write.mode("overwrite").parquet(predictions_path)
else:
    # Load saved predictions
    print("Loading saved predictions...")
    test_predictions_gbt = spark.read.parquet(predictions_path)

# Metrics calculations for GBT
test_auc_gbt = evaluator.evaluate(test_predictions_gbt)

# Confusion matrix calculations for GBT
tp = test_predictions_gbt.filter((col("label") == 1) & (col("prediction") == 1)).count()
tn = test_predictions_gbt.filter((col("label") == 0) & (col("prediction") == 0)).count()
fp = test_predictions_gbt.filter((col("label") == 0) & (col("prediction") == 1)).count()
fn = test_predictions_gbt.filter((col("label") == 1) & (col("prediction") == 0)).count()

# Metrics derived from confusion matrix
sensitivity_gbt = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity_gbt = tn / (tn + fp) if (tn + fp) > 0 else 0
precision_gbt = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_gbt = sensitivity_gbt
f1_gbt = 2 * (precision_gbt * recall_gbt) / (precision_gbt + recall_gbt) if (precision_gbt + recall_gbt) > 0 else 0
accuracy_gbt = (tp + tn) / (tp + tn + fp + fn)

# Print evaluation metrics for GBT
print(f"GBT Test AUC: {test_auc_gbt}")
print(f"Sensitivity (Recall): {sensitivity_gbt}")
print(f"Specificity: {specificity_gbt}")
print(f"Precision: {precision_gbt}")
print(f"Recall: {recall_gbt}")
print(f"F1 Score: {f1_gbt}")
print(f"Accuracy: {accuracy_gbt}")

# Print confusion matrix
print("\nConfusion Matrix:")
print(f"True Positives (TP): {tp}")
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")

In [13]:
import os
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col

# Define the GBT model with optimized parameters
gbt = GBTClassifier(
    labelCol="label", 
    featuresCol="features_scaled", 
    maxIter=20,  # Increased initial value to allow more trees
    maxDepth=30,  # Reduce to avoid overfitting
    stepSize=0.1,  # Lower learning rate for smoother updates
    subsamplingRate=0.8,  # Introduce some randomness
    maxBins=32  # Helps with categorical features
)

# Define a global binary evaluator for areaUnderROC
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Adaptive Early Stopping Function for GBT
def adaptive_early_stopping(classifier, train_data, valid_data, patience=5, max_iter_increment=10, max_iter_cap=40):
    """
    Train a classifier with adaptive early stopping by incrementally increasing maxIter.
    """
    best_model = None
    best_auc = 0.0
    best_iter = 0
    patience_counter = 0
    current_max_iter = classifier.getMaxIter()
    
    while current_max_iter <= max_iter_cap:
        classifier.setMaxIter(current_max_iter)
        print(f"\nTraining with maxIter={current_max_iter}...")
        
        model = classifier.fit(train_data)
        valid_predictions = model.transform(valid_data)
        validation_auc = evaluator.evaluate(valid_predictions)
        
        print(f"Validation AUC with maxIter={current_max_iter}: {validation_auc}")
        
        if validation_auc > best_auc:
            best_auc = validation_auc
            best_model = model
            best_iter = current_max_iter
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Early stopping condition
        if patience_counter >= patience:
            print(f"Early stopping triggered. Best maxIter: {best_iter} with Validation AUC: {best_auc}")
            break
        
        # Increment maxIter for the next iteration
        current_max_iter += max_iter_increment
    
    return best_model, best_auc

# Define predictions path
predictions_path = "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/checkpoint4"

# Check if the directory exists
if not os.path.exists(predictions_path):
    # Training and Adaptive Early Stopping
    print("\nTraining GBT with Adaptive Early Stopping...")
    best_gbt_model, best_gbt_auc = adaptive_early_stopping(
        gbt, train_data_balanced, valid_data, patience=3, max_iter_increment=10, max_iter_cap=40
    )
    print(f"Best AUC for GBT: {best_gbt_auc}")

    # Evaluate the best model on test data
    test_predictions_gbt = best_gbt_model.transform(test_data)

    # Save predictions to Parquet
    test_predictions_gbt.write.mode("overwrite").parquet(predictions_path)
else:
    # Load saved predictions
    print("Loading saved predictions...")
    test_predictions_gbt = spark.read.parquet(predictions_path)

# Metrics calculations for GBT
test_auc_gbt = evaluator.evaluate(test_predictions_gbt)

# Confusion matrix calculations for GBT
tp = test_predictions_gbt.filter((col("label") == 1) & (col("prediction") == 1)).count()
tn = test_predictions_gbt.filter((col("label") == 0) & (col("prediction") == 0)).count()
fp = test_predictions_gbt.filter((col("label") == 0) & (col("prediction") == 1)).count()
fn = test_predictions_gbt.filter((col("label") == 1) & (col("prediction") == 0)).count()

# Metrics derived from confusion matrix
sensitivity_gbt = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity_gbt = tn / (tn + fp) if (tn + fp) > 0 else 0
precision_gbt = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_gbt = sensitivity_gbt
f1_gbt = 2 * (precision_gbt * recall_gbt) / (precision_gbt + recall_gbt) if (precision_gbt + recall_gbt) > 0 else 0
accuracy_gbt = (tp + tn) / (tp + tn + fp + fn)

# Print evaluation metrics for GBT
print(f"GBT Test AUC: {test_auc_gbt}")
print(f"Sensitivity (Recall): {sensitivity_gbt}")
print(f"Specificity: {specificity_gbt}")
print(f"Precision: {precision_gbt}")
print(f"Recall: {recall_gbt}")
print(f"F1 Score: {f1_gbt}")
print(f"Accuracy: {accuracy_gbt}")

# Print confusion matrix
print("\nConfusion Matrix:")
print(f"True Positives (TP): {tp}")
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")


Training GBT with Adaptive Early Stopping...

Training with maxIter=20...
Validation AUC with maxIter=20: 0.8616491944038756

Training with maxIter=30...
Validation AUC with maxIter=30: 0.8694101762898649

Training with maxIter=40...
Validation AUC with maxIter=40: 0.8738416467234345
Best AUC for GBT: 0.8738416467234345
GBT Test AUC: 0.8604922596596625
Sensitivity (Recall): 0.6163398692810458
Specificity: 0.9390956928473981
Precision: 0.6014030612244898
Recall: 0.6163398692810458
F1 Score: 0.6087798579728858
Accuracy: 0.8972184531886025

Confusion Matrix:
True Positives (TP): 943
True Negatives (TN): 9637
False Positives (FP): 625
False Negatives (FN): 587


In [48]:
import os
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col

# Define the GBT model with optimized parameters
gbt = GBTClassifier(
    labelCol="label", 
    featuresCol="features_scaled", 
    maxIter=15,  # Initial iteration count
    maxDepth=8,  # Reduced depth for post-pruning
    stepSize=0.1,  # Lower learning rate for smoother updates
    subsamplingRate=0.8,  # Introduce randomness
    maxBins=32  # Helps with categorical features
)

# Define a global binary evaluator for areaUnderROC
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Adaptive Post-Pruning Function for GBT
def adaptive_post_pruning(classifier, train_data, valid_data, patience=3, max_iter_increment=10, max_iter_cap=100):
    """
    Train a classifier with adaptive post-pruning by stopping when validation performance plateaus.
    """
    best_model = None
    best_auc = 0.0
    best_depth = classifier.getMaxDepth()
    patience_counter = 0
    current_depth = classifier.getMaxDepth()
    
    while patience_counter < patience and current_depth > 2:
        classifier.setMaxDepth(current_depth)
        print(f"\nTraining with maxDepth={current_depth}...")
        
        model = classifier.fit(train_data)
        valid_predictions = model.transform(valid_data)
        validation_auc = evaluator.evaluate(valid_predictions)
        
        print(f"Validation AUC with maxDepth={current_depth}: {validation_auc}")
        
        if validation_auc > best_auc:
            best_auc = validation_auc
            best_model = model
            best_depth = current_depth
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Reduce tree depth for post-pruning
        current_depth -= 1
    
    print(f"Post-pruning completed. Best maxDepth: {best_depth} with Validation AUC: {best_auc}")
    return best_model, best_auc

# Define predictions path
predictions_path = "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/checkpoint10"

# Check if the directory exists
if not os.path.exists(predictions_path):
    # Training and Adaptive Post-Pruning
    print("\nTraining GBT with Adaptive Post-Pruning...")
    best_gbt_model, best_gbt_auc = adaptive_post_pruning(gbt, train_data_balanced, valid_data)
    print(f"Best AUC for GBT: {best_gbt_auc}")

    # Evaluate the best model on test data
    test_predictions_gbt = best_gbt_model.transform(test_data)

    # Save predictions to Parquet
    test_predictions_gbt.write.mode("overwrite").parquet(predictions_path)
else:
    # Load saved predictions
    print("Loading saved predictions...")
    test_predictions_gbt = spark.read.parquet(predictions_path)

# Metrics calculations for GBT
test_auc_gbt = evaluator.evaluate(test_predictions_gbt)

# Confusion matrix calculations for GBT
tp = test_predictions_gbt.filter((col("label") == 1) & (col("prediction") == 1)).count()
tn = test_predictions_gbt.filter((col("label") == 0) & (col("prediction") == 0)).count()
fp = test_predictions_gbt.filter((col("label") == 0) & (col("prediction") == 1)).count()
fn = test_predictions_gbt.filter((col("label") == 1) & (col("prediction") == 0)).count()

# Metrics derived from confusion matrix
sensitivity_gbt = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity_gbt = tn / (tn + fp) if (tn + fp) > 0 else 0
precision_gbt = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_gbt = sensitivity_gbt
f1_gbt = 2 * (precision_gbt * recall_gbt) / (precision_gbt + recall_gbt) if (precision_gbt + recall_gbt) > 0 else 0
accuracy_gbt = (tp + tn) / (tp + tn + fp + fn)

# Print evaluation metrics for GBT
print(f"GBT Test AUC: {test_auc_gbt}")
print(f"Sensitivity (Recall): {sensitivity_gbt}")
print(f"Specificity: {specificity_gbt}")
print(f"Precision: {precision_gbt}")
print(f"Recall: {recall_gbt}")
print(f"F1 Score: {f1_gbt}")
print(f"Accuracy: {accuracy_gbt}")

# Print confusion matrix
print("\nConfusion Matrix:")
print(f"True Positives (TP): {tp}")
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")


Training GBT with Adaptive Post-Pruning...

Training with maxDepth=8...
Validation AUC with maxDepth=8: 0.9013993388988732

Training with maxDepth=7...
Validation AUC with maxDepth=7: 0.9018683240785833

Training with maxDepth=6...
Validation AUC with maxDepth=6: 0.8941122256891536

Training with maxDepth=5...
Validation AUC with maxDepth=5: 0.8858796917608004

Training with maxDepth=4...
Validation AUC with maxDepth=4: 0.8782419112103802
Post-pruning completed. Best maxDepth: 7 with Validation AUC: 0.9018683240785833
Best AUC for GBT: 0.9018683240785833
GBT Test AUC: 0.8986112225699728
Sensitivity (Recall): 0.7522875816993464
Specificity: 0.8581173260572987
Precision: 0.4415036440352896
Recall: 0.7522875816993464
F1 Score: 0.5564418660865361
Accuracy: 0.8443860244233379

Confusion Matrix:
True Positives (TP): 1151
True Negatives (TN): 8806
False Positives (FP): 1456
False Negatives (FN): 379


In [25]:
import os
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col

# Define the GBT model with pre-pruning parameters
gbt = GBTClassifier(
    labelCol="label", 
    featuresCol="features_scaled", 
    maxIter=20,  # Set a fixed iteration count
    maxDepth=30,  # Pre-pruning by limiting depth
    stepSize=0.1,  # Learning rate
    subsamplingRate=0.8,  # Introduce randomness
    maxBins=32  # Helps with categorical features
)

# Define a global binary evaluator for areaUnderROC
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Train the pre-pruned GBT model
print("\nTraining GBT with Pre-Pruning...")
gbt_model = gbt.fit(train_data_balanced)

# Validate the model
valid_predictions = gbt_model.transform(valid_data)
valid_auc = evaluator.evaluate(valid_predictions)
print(f"Validation AUC with Pre-Pruning: {valid_auc}")

# Define predictions path
predictions_path = "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/checkpoint11"

# Check if the directory exists
if not os.path.exists(predictions_path):
    # Evaluate the model on test data
    test_predictions_gbt = gbt_model.transform(test_data)
    
    # Save predictions to Parquet
    test_predictions_gbt.write.mode("overwrite").parquet(predictions_path)
else:
    # Load saved predictions
    print("Loading saved predictions...")
    test_predictions_gbt = spark.read.parquet(predictions_path)

# Metrics calculations for GBT
test_auc_gbt = evaluator.evaluate(test_predictions_gbt)

# Confusion matrix calculations
tp = test_predictions_gbt.filter((col("label") == 1) & (col("prediction") == 1)).count()
tn = test_predictions_gbt.filter((col("label") == 0) & (col("prediction") == 0)).count()
fp = test_predictions_gbt.filter((col("label") == 0) & (col("prediction") == 1)).count()
fn = test_predictions_gbt.filter((col("label") == 1) & (col("prediction") == 0)).count()

# Metrics derived from confusion matrix
sensitivity_gbt = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity_gbt = tn / (tn + fp) if (tn + fp) > 0 else 0
precision_gbt = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_gbt = sensitivity_gbt
f1_gbt = 2 * (precision_gbt * recall_gbt) / (precision_gbt + recall_gbt) if (precision_gbt + recall_gbt) > 0 else 0
accuracy_gbt = (tp + tn) / (tp + tn + fp + fn)

# Print evaluation metrics
print(f"GBT Test AUC: {test_auc_gbt}")
print(f"Sensitivity (Recall): {sensitivity_gbt}")
print(f"Specificity: {specificity_gbt}")
print(f"Precision: {precision_gbt}")
print(f"Recall: {recall_gbt}")
print(f"F1 Score: {f1_gbt}")
print(f"Accuracy: {accuracy_gbt}")

# Print confusion matrix
print("\nConfusion Matrix:")
print(f"True Positives (TP): {tp}")
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")


Training GBT with Pre-Pruning...
Validation AUC with Pre-Pruning: 0.8581909442958182
GBT Test AUC: 0.8519167421402406
Sensitivity (Recall): 0.6300653594771242
Specificity: 0.9320795166634185
Precision: 0.580373269114991
Recall: 0.6300653594771242
F1 Score: 0.6041993105609527
Accuracy: 0.892893487109905

Confusion Matrix:
True Positives (TP): 964
True Negatives (TN): 9565
False Positives (FP): 697
False Negatives (FN): 566


In [36]:
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col
import os

# Define the GBT model with adjusted parameters
gbt = GBTClassifier(
    labelCol="label", 
    featuresCol="features_scaled", 
    maxIter=10, 
    maxDepth= 7,           # Increased depth to allow more complex trees
    minInstancesPerNode=10,  # Minimum number of samples per node to avoid overfitting
    subsamplingRate=0.8,    # Subsample rate to control variance and prevent overfitting
    stepSize=0.1           # Control the learning rate for gradient boosting
)

# Define a global binary evaluator for areaUnderROC
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Adaptive Early Stopping Function for GBT
def adaptive_early_stopping(classifier, train_data, valid_data, patience=3, max_iter_increment=10, max_iter_cap=100):
    best_model = None
    best_auc = 0.0
    best_iter = 0
    patience_counter = 0
    current_max_iter = max_iter_increment

    while current_max_iter <= max_iter_cap:
        classifier.setMaxIter(current_max_iter)
        print(f"\nTraining with maxIter={current_max_iter}...")

        model = classifier.fit(train_data)
        valid_predictions = model.transform(valid_data)
        validation_auc = evaluator.evaluate(valid_predictions)

        print(f"Validation AUC with maxIter={current_max_iter}: {validation_auc}")

        # Capture the model with the best AUC
        if validation_auc > best_auc:
            best_auc = validation_auc
            best_model = model
            best_iter = current_max_iter
            patience_counter = 0
        else:
            patience_counter += 1

        # Early stopping condition (stop earlier if no improvement)
        if patience_counter >= patience:
            print(f"Early stopping triggered. Best maxIter: {best_iter} with Validation AUC: {best_auc}")
            break

        # Increment maxIter for the next iteration
        current_max_iter += max_iter_increment

    return best_model, best_auc

# Define predictions path
predictions_path = "file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/checkpoint4"

# Check if the directory exists
if not os.path.exists(predictions_path):
    # Training and Adaptive Early Stopping
    print("\nTraining GBT with Adaptive Early Stopping...")
    best_gbt_model, best_gbt_auc = adaptive_early_stopping(
        gbt, train_data_balanced, valid_data, patience=3, max_iter_increment=10, max_iter_cap=100
    )
    print(f"Best AUC for GBT: {best_gbt_auc}")

    # Evaluate the best model on test data
    test_predictions_gbt = best_gbt_model.transform(test_data)

    # Save predictions to Parquet
    test_predictions_gbt.write.mode("overwrite").parquet(predictions_path)
else:
    # Load saved predictions
    print("Loading saved predictions...")
    test_predictions_gbt = spark.read.parquet(predictions_path)

# Metrics calculations for GBT
test_auc_gbt = evaluator.evaluate(test_predictions_gbt)

# Confusion matrix calculations for GBT
tp = test_predictions_gbt.filter((col("label") == 1) & (col("prediction") == 1)).count()
tn = test_predictions_gbt.filter((col("label") == 0) & (col("prediction") == 0)).count()
fp = test_predictions_gbt.filter((col("label") == 0) & (col("prediction") == 1)).count()
fn = test_predictions_gbt.filter((col("label") == 1) & (col("prediction") == 0)).count()

# Metrics derived from confusion matrix
sensitivity_gbt = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity_gbt = tn / (tn + fp) if (tn + fp) > 0 else 0
precision_gbt = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_gbt = sensitivity_gbt
f1_gbt = 2 * (precision_gbt * recall_gbt) / (precision_gbt + recall_gbt) if (precision_gbt + recall_gbt) > 0 else 0
accuracy_gbt = (tp + tn) / (tp + tn + fp + fn)

# Print evaluation metrics for GBT
print(f"GBT Test AUC: {test_auc_gbt}")
print(f"Sensitivity (Recall): {sensitivity_gbt}")
print(f"Specificity: {specificity_gbt}")
print(f"Precision: {precision_gbt}")
print(f"Recall: {recall_gbt}")
print(f"F1 Score: {f1_gbt}")
print(f"Accuracy: {accuracy_gbt}")

# Print confusion matrix
print("\nConfusion Matrix:")
print(f"True Positives (TP): {tp}")
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")


Training GBT with Adaptive Early Stopping...

Training with maxIter=10...
Validation AUC with maxIter=10: 0.891357174870278

Training with maxIter=20...
Validation AUC with maxIter=20: 0.9084158821833926

Training with maxIter=30...
Validation AUC with maxIter=30: 0.9159154880327568

Training with maxIter=40...
Validation AUC with maxIter=40: 0.9200972463181741

Training with maxIter=50...
Validation AUC with maxIter=50: 0.923199065752399

Training with maxIter=60...
Validation AUC with maxIter=60: 0.9252813886862651

Training with maxIter=70...
Validation AUC with maxIter=70: 0.9274140963896049

Training with maxIter=80...
Validation AUC with maxIter=80: 0.9287928325833175

Training with maxIter=90...
Validation AUC with maxIter=90: 0.929585000300756

Training with maxIter=100...
Validation AUC with maxIter=100: 0.9301843142994508
Best AUC for GBT: 0.9301843142994508
GBT Test AUC: 0.9265574943028619
Sensitivity (Recall): 0.7738562091503268
Specificity: 0.9336386669265251
Precision: 0

In [59]:
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col, when, lit
import os

# Define the GBT model
gbt = GBTClassifier(
    labelCol="label",
    featuresCol="features_scaled",
    maxIter=12,
    maxDepth=7,
    minInstancesPerNode=10,
    subsamplingRate=0.8,
    stepSize=0.1
)

# Define evaluator
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Function to train with early stopping
def adaptive_early_stopping(classifier, train_data, valid_data, patience=3, max_iter_increment=10, max_iter_cap=100):
    best_model = None
    best_auc = 0.0
    best_iter = 0
    patience_counter = 0
    current_max_iter = max_iter_increment

    while current_max_iter <= max_iter_cap:
        classifier.setMaxIter(current_max_iter)
        print(f"\nTraining with maxIter={current_max_iter}...")

        model = classifier.fit(train_data)
        valid_predictions = model.transform(valid_data)
        validation_auc = evaluator.evaluate(valid_predictions)

        print(f"Validation AUC with maxIter={current_max_iter}: {validation_auc}")

        if validation_auc > best_auc:
            best_auc = validation_auc
            best_model = model
            best_iter = current_max_iter
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping triggered. Best maxIter: {best_iter} with Validation AUC: {best_auc}")
            break

        current_max_iter += max_iter_increment

    return best_model, best_auc

# Training and Adaptive Early Stopping
print("\nTraining GBT with Adaptive Early Stopping...")
best_gbt_model, best_gbt_auc = adaptive_early_stopping(
    gbt, train_data_balanced, valid_data, patience=3, max_iter_increment=10, max_iter_cap=100
)
print(f"Best AUC for GBT: {best_gbt_auc}")

# Evaluate the best model on test data
test_predictions = best_gbt_model.transform(test_data)

# Define thresholds to evaluate
thresholds = [i / 10 for i in range(1, 10)]  # 0.1 to 0.9
best_threshold = 0.5
best_f1 = 0.0
metrics_dict = {}

# Iterate over different thresholds
for threshold in thresholds:
    print(f"\nEvaluating threshold: {threshold}")

    # Apply threshold to get predictions
    predictions_with_threshold = test_predictions.withColumn(
        "thresholded_prediction",
        when(col("probability")[1] >= lit(threshold), lit(1)).otherwise(lit(0))
    )

    # Compute confusion matrix
    tp = predictions_with_threshold.filter((col("label") == 1) & (col("thresholded_prediction") == 1)).count()
    tn = predictions_with_threshold.filter((col("label") == 0) & (col("thresholded_prediction") == 0)).count()
    fp = predictions_with_threshold.filter((col("label") == 0) & (col("thresholded_prediction") == 1)).count()
    fn = predictions_with_threshold.filter((col("label") == 1) & (col("thresholded_prediction") == 0)).count()

    # Calculate metrics
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = sensitivity
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    # Store metrics for this threshold
    metrics_dict[threshold] = {
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "accuracy": accuracy
    }

    print(f"Threshold: {threshold} - F1 Score: {f1_score}, Precision: {precision}, Recall: {recall}")

    # Update best threshold based on F1-score
    if f1_score > best_f1:
        best_f1 = f1_score
        best_threshold = threshold

# Display best threshold with metrics
best_metrics = metrics_dict[best_threshold]
print(f"\nBest Threshold: {best_threshold}")
print(f"Sensitivity (Recall): {best_metrics['sensitivity']}")
print(f"Specificity: {best_metrics['specificity']}")
print(f"Precision: {best_metrics['precision']}")
print(f"Recall: {best_metrics['recall']}")
print(f"F1 Score: {best_metrics['f1_score']}")
print(f"Accuracy: {best_metrics['accuracy']}")


Training GBT with Adaptive Early Stopping...

Training with maxIter=10...
Validation AUC with maxIter=10: 0.8904720834959404

Training with maxIter=20...
Validation AUC with maxIter=20: 0.906590156724239

Training with maxIter=30...
Validation AUC with maxIter=30: 0.9140223541772577

Training with maxIter=40...
Validation AUC with maxIter=40: 0.9186188302913442

Training with maxIter=50...
Validation AUC with maxIter=50: 0.9218647734133516

Training with maxIter=60...
Validation AUC with maxIter=60: 0.9250763383339408

Training with maxIter=70...
Validation AUC with maxIter=70: 0.9268910052967434

Training with maxIter=80...
Validation AUC with maxIter=80: 0.9290086669875383

Training with maxIter=90...
Validation AUC with maxIter=90: 0.9302988302758507

Training with maxIter=100...
Validation AUC with maxIter=100: 0.9311703689382342
Best AUC for GBT: 0.9311703689382342

Evaluating threshold: 0.1


AnalysisException: "Can't extract value from probability#10412954: need struct type but got struct<type:tinyint,size:int,indices:array<int>,values:array<double>>;"